# Sanity checks & verifier debugging

Small-scale local checks for the metric-repair algorithms, plus diagnostics for the
`left_edge_heuristic` / `pivot_heuristic` vs `verifier` failures (`valid == 0`).

Run top to bottom.
- **§1** quick sanity sweep (the small-scale experiment).
- **§2–§5** localize *whether the verifier or the heuristics are at fault*.

The key idea: `is_valid_cover` (§0) is an **independent** ground-truth cover test, so any
disagreement with `verifier` points at the verifier; agreement points at the heuristics.

In [ ]:
%run Packages_and_Functions.ipynb
import pandas as pd

ALGOS = {
    'domr_alg': domr_alg,
    'shortest_path_cover': shortest_path_cover,
    'left_edge_heuristic': left_edge_heuristic,
    'pivot_heuristic': pivot_heuristic,
    'l1_min_heuristic': l1_min_heuristic,
}

def is_valid_cover(G, S, tol=1e-9):
    '''Ground-truth cover test, INDEPENDENT of verifier(): delete the cover edges, then every
    remaining (non-cover) edge must NOT be broken by a path among the remaining edges.
    Returns (is_valid, witness) where witness = (u, v, w, dist) of a broken non-cover edge.'''
    Sset = set(S) | set((b, a) for (a, b) in S)
    H = G.copy()
    for (a, b) in S:
        if H.has_edge(a, b):
            H.delete_edge(a, b)
    D = H.distance_all_pairs(by_weight=True)
    for u, v, w in G.edges(sort=True):
        if (u, v) in Sset:
            continue
        duv = D.get(u, {}).get(v, float('inf'))
        if duv < float(w) - tol:
            return False, (u, v, float(w), float(duv))
    return True, None

print('loaded; algorithms:', list(ALGOS))

## 1. Quick sanity sweep

Tiny grids, a few reps. Mean cover size and validity per algorithm — the everyday sanity check
before launching anything bigger.

In [ ]:
rows = []
for n in [8, 12]:
    for p in [0.3, 0.5]:
        for rep in range(5):
            seed_all(1000 * int(n) + int(100 * p) + rep)
            G = random_geometric_weighted_graph(n, p)
            if not G.is_connected():
                continue
            for name, f in ALGOS.items():
                S = f(G)
                rows.append({'n': int(n), 'p': float(p), 'rep': rep, 'algo': name,
                             'cover': len(S), 'valid': int(verifier(G, S))})
df = pd.DataFrame(rows)
print(df.groupby('algo')[['cover', 'valid']].mean())
df.head(int(20))

## 2. Verifier litmus test

`domr_alg(G)` returns every broken edge — a **decrease-only** cover, which is ALWAYS a valid
general-repair cover (just lower those edges to their shortest-path distance).

**If `verifier` ever rejects a `domr` cover, the verifier is definitively wrong.** Expect 0.

In [ ]:
seed_all(0)
fails, total = 0, 0
for t in range(300):
    G = random_geometric_weighted_graph(10, 0.5)
    if not G.is_connected():
        continue
    total += 1
    if verifier(G, domr_alg(G)) != 1:
        fails += 1
        if fails <= 3:
            print('verifier REJECTED a domr (valid) cover at t =', t)
print('domr covers rejected by verifier:', fails, '/', total, '  (expect 0 if verifier is correct)')

## 3. Verifier vs independent ground truth

Compare `verifier` to `is_valid_cover` on every algorithm's output.
- **0 disagreements** → verifier is correct, so any `valid==0` is a *genuinely invalid cover* (bug is in the heuristic).
- **disagreements** → the verifier is the problem; the witness shows the offending edge.

In [ ]:
seed_all(1)
disagree, total = 0, 0
for t in range(300):
    G = random_geometric_weighted_graph(11, 0.5)
    if not G.is_connected():
        continue
    total += 1
    for name, f in ALGOS.items():
        S = f(G)
        v = bool(verifier(G, S))
        gt, witness = is_valid_cover(G, S)
        if v != gt:
            disagree += 1
            if disagree <= 6:
                print('DISAGREE t=%d %s: verifier=%s truth=%s witness=%s' % (t, name, v, gt, witness))
print('verifier vs ground-truth disagreements:', disagree, 'over', total, 'graphs x', len(ALGOS), 'algos')

## 4. Dissect a failing case

For the first `left_edge` / `pivot` failure, separate the two possible causes:
- **`verifier(Kn, full)` fails** → the heuristic didn't even produce a valid cover for the
  *completion* `Kn` (e.g. Gilbert's single `k`-pass left residual triangle violations).
- **`verifier(Kn, full)` passes but the reduced cover fails on `G`** → `reduce_solution` lost coverage.

In [ ]:
def dissect(which):
    seed_all(2)
    for t in range(2000):
        G = random_geometric_weighted_graph(11, 0.5)
        if not G.is_connected():
            continue
        S = ALGOS[which](G)
        if verifier(G, S) == 0:
            gt, witness = is_valid_cover(G, S)
            Kn = complete(G)
            full = Gilbert_Jain_IOMR(Kn) if which == 'left_edge_heuristic' else MVD_Pivot(Kn)
            print('=== %s  (t=%d) ===' % (which, t))
            print(' reduced cover |S| :', len(S))
            print(' ground-truth valid:', gt, '| broken witness (u,v,w,dist):', witness)
            print(' completion already metric? ', is_metric(Kn))
            print(' full cover |full| :', len(full), '| verifier(Kn, full):', verifier(Kn, full),
                  '| is_metric after full-cover fix? (see note)')
            return G, S, Kn, full
    print('no failing case found for', which)
    return None

_ = dissect('left_edge_heuristic')
print()
_ = dissect('pivot_heuristic')

## 5. Integer vs float weights (exact-comparison fragility)

`verifier` (and `is_metric`, `domr_alg`) test brokenness with **exact** float comparison
(`D[u][v] != w`). For non-integer weights, summing a tie-path can round to *slightly less* than the
direct edge → a spurious "broken" → spurious `valid==0`. This isolates that effect: geometric
weights are integers (exact), Euclidean weights are floats.

In [ ]:
def disagreements(gen, label, trials=200, n=11, p=0.5):
    seed_all(7)
    d, total = 0, 0
    for _ in range(trials):
        G = gen(n, p)
        if not G.is_connected():
            continue
        total += 1
        for f in ALGOS.values():
            S = f(G)
            if bool(verifier(G, S)) != is_valid_cover(G, S)[0]:
                d += 1
    print('%-28s %3d disagreements over %d connected graphs' % (label, d, total))

disagreements(random_geometric_weighted_graph, 'geometric (int weights)')
disagreements(random_metric_graph, 'metric / Euclidean (float)')

## 6. General vs IOMR verifier

`iomr_verifier` checks the **increase-only** model: same as `verifier`, but it also requires the
cover edges themselves to be unbroken by frozen detours (increase-only can't lower them).

IOMR is a *special case* of general repair, so `iomr_verifier` implies `verifier`. The table should
show `iomr <= general` per algorithm, and **"IOMR-accepts-but-general-rejects" must be 0** — the
empirical confirmation that switching to IOMR can't rescue covers the general verifier already rejects.

In [ ]:
seed_all(11)
rows = []
for t in range(200):
    G = random_geometric_weighted_graph(11, 0.5)
    if not G.is_connected():
        continue
    for name, f in ALGOS.items():
        S = f(G)
        rows.append({'algo': name,
                     'general': int(verifier(G, S)),
                     'iomr': int(iomr_verifier(G, S))})
res = pd.DataFrame(rows)
print(res.groupby('algo')[['general', 'iomr']].mean())
viol = res[(res['iomr'] == 1) & (res['general'] == 0)]
print('IOMR-accepts-but-general-rejects (must be 0):', len(viol))

## 7. The pivot / left-edge cover bug: matrix position vs vertex label

`pivot_heuristic = reduce_solution(MVD_Pivot(complete(G)), G)`. Two distinct bugs lived in this path;
both are now fixed in `metric_repair.sage`:

1. **Asymmetry** — the vectorized recursion wrote `X[j,k]` but not `X[k,j]`, so later pivots read
   stale distances. The paper's `x` is a vector over *unordered* pairs (symmetric), so this was a real
   deviation. Fixed by writing both entries; §8 confirms the output is then always a metric.
2. **Position-vs-label** — `MVD_Pivot` and `Gilbert_Jain_IOMR` build a NumPy matrix from
   `weighted_adjacency_matrix` (indexed by **position** `0..n-1`) and recorded the cover in those
   positions. But the generators build the graph from its edge list, so isolated vertices are dropped
   and `Kn`'s vertex **labels** are *not* `0..n-1`. The verifier then read position pairs as label
   pairs and rejected valid covers — worse on sparser graphs (more dropped vertices), which is why
   n=20, p=0.2 surfaced it. Fixed by mapping positions back to `Kn.vertices(sort=True)` before return.

The cell below exercises the **public** functions in a regime that triggers non-contiguous labels.
Mind the repair model: `MVD_Pivot` solves **general** MR (it may *decrease* edges), so it's checked
against `verifier` only and deliberately fails `iomr_verifier`. `Gilbert_Jain_IOMR` (left-edge) is
**increase-only**, so it's additionally checked against the stronger `iomr_verifier`. All counts 0.

In [ ]:
seed_all(3)
piv_bad, le_bad, le_iomr_bad, noncontig, total = 0, 0, 0, 0, 0
for t in range(200):
    G = random_geometric_weighted_graph(20, 0.2)   # sparse => isolated vertices => label gaps
    if not G.is_connected():
        continue
    total += 1
    Kn = complete(G)
    if Kn.vertices(sort=True) != list(range(Kn.num_verts())):
        noncontig += 1
    if verifier(Kn, MVD_Pivot(Kn)) == 0:            # pivot solves GENERAL MR
        piv_bad += 1
    le = Gilbert_Jain_IOMR(Kn)                       # left-edge solves INCREASE-ONLY (IOMR) MR
    if verifier(Kn, le) == 0:
        le_bad += 1
    if iomr_verifier(Kn, le) == 0:                  # ...so also valid in the stronger IOMR model
        le_iomr_bad += 1
print('non-contiguous-label graphs:', noncontig, '/', total, '  (the cases the old code mis-mapped)')
print('MVD_Pivot invalid (general verifier):     ', piv_bad, '/', total, '  (expect 0)')
print('Gilbert_Jain invalid (general verifier):  ', le_bad, '/', total, '  (expect 0)')
print('Gilbert_Jain invalid (IOMR verifier):     ', le_iomr_bad, '/', total, '  (expect 0; increase-only)')

## 8. Is MVD_Pivot's OUTPUT matrix a metric?

By the projection argument, Algorithm 1's clamps preserve every earlier pivot's triangle, so the
modified `x` is always a **metric**. This checks the output matrix `X` directly. It is *orthogonal* to
§7: it isolates the **algorithm** (the matrix) from the **cover bookkeeping** (the position→label
mapping where §7's bug lived). Expect **0** — and §8 was already 0 while §7 still failed, which is
exactly what pinned the residual bug to the cover's labels rather than to the algorithm.

In [ ]:
def matrix_metric_violation(X, tol=1e-9):
    '''Return a violating triple (a, b, c) of the symmetric matrix X, or None if X is a metric.'''
    n = X.shape[0]
    for a in range(n):
        for b in range(a + 1, n):
            for c in range(n):
                if c == a or c == b:
                    continue
                if X[a, b] > X[a, c] + X[c, b] + tol:
                    return (a, b, c)
    return None

seed_all(3)
not_metric, total = 0, 0
for t in range(200):
    G = random_geometric_weighted_graph(20, 0.2)
    if not G.is_connected():
        continue
    total += 1
    X = np.array(complete(G).weighted_adjacency_matrix(), dtype=float)
    S = set()
    _mvd_pivot_rec(list(range(X.shape[0])), X, S)   # low-level: inspect the working matrix itself
    bad = matrix_metric_violation(X)
    if bad is not None:
        not_metric += 1
        if not_metric <= 3:
            a, b, c = bad
            print('x NOT a metric at t=%d:  x(%d,%d)=%g  >  x(%d,%d)=%g + x(%d,%d)=%g  | symmetric=%s'
                  % (t, a, b, X[a, b], a, c, X[a, c], c, b, X[c, b], np.allclose(X, X.T)))
print('MVD output matrix NOT a metric:', not_metric, '/', total,
      '  (0 => algorithm faithful; the §7 cover bug was the position->label mapping)')